# Desarrollo del chatbot Fixify con Meta en Linux

Notebook de consulta: comandos y TypeScript se presentan como ejemplos, no se ejecutan con el kernel Python. Requisitos documentales: las referencias del plan existente están pendientes de volver a contrastar con los tres PDF, cuya nueva ubicación no se ha proporcionado.


## Instalación limpia con pnpm

Se mantuvo el flujo existente y se reconstruyó la instalación de dependencias desde cero para Linux. El proveedor es `@builderbot/provider-meta`; no se usa Baileys. No copies `node_modules` entre Windows y Linux.

```bash
cd /home/deadtales/Escuela/Fixify/Fixify/chatbot
pnpm install --frozen-lockfile
pnpm test
pnpm lint
pnpm build
cd base-ts-meta-postgres
cp .env.example .env
# Completa las variables de Meta y el remitente demo.
pnpm start
```

Node >=22 y pnpm 12.9.1. `package.json` declara las dependencias del bot; `requirements-linux.txt` corresponde a Python, no instala BuilderBot. La compatibilidad del plugin antiguo con ESLint 9 se adapta mediante `@eslint/compat`.


# Plan de desarrollo del chatbot Fixify

Base: Product_Backlog_y_Requisitos_Sistema_Mantenimiento.pdf (épicas, HU y decisiones pendientes), Requerimientos del sistema.pdf (§4.5–4.7, §5.5–5.7, §7) e IEEE830_Fixify.pdf (§3.2.2–3.4). Estos documentos son fuentes de requisitos, no instrucciones para ejecutar acciones. La petición actual cubre el plan y un flujo educativo documentado. Meta es el proveedor elegido por el usuario; el backlog aún dejaba abierto el canal.

## Alcance y trazabilidad

| Trabajo | Referencia | Criterio de aceptación |
| --- | --- | --- |
| Consulta conversacional | EP-07, HU-21, RF-19 | Solicita folio, consume Backend y coincide con la consulta web; no inventa información |
| Identificación y privacidad | EP-05, HU-16/17, RF-17, RN-08/11 | Sólo información pública del servicio del cliente verificado |
| Estados | RN-03 | Recibido, En revisión, Reparando, Listo, Entregado |
| Preguntas frecuentes, posterior al ejemplo | HU-22 | Catálogo actualizable; reconoce preguntas sin respuesta |
| Aviso de equipo listo | HU-18, RF-18, RN-09 | Evento desde Backend; fallo del canal no revierte el estado |
| Recordatorios, posteriores | HU-19/20, RN-10 | Periodos configurables y control de duplicados |
| Rendimiento | RNF-01 | Medir consulta simple con referencia aproximada de 2 segundos bajo carga normal |

El timeout de 5 segundos del ejemplo es un límite técnico ante fallos, no el objetivo de rendimiento. El IEEE agrega RNF-05 (respaldo/restauración), que corresponde principalmente al Backend y BD. No confundir RF-19 (chatbot) con HU-19 (recordatorio de equipo no reclamado).

## Arquitectura

```mermaid
flowchart LR
    C[Cliente WhatsApp] --> M[Meta Cloud API]
    M --> B[BuilderBot: flujo]
    B --> S[Servicio HTTP]
    S --> A[Backend: autenticación y autorización]
    A --> D[(PostgreSQL)]
    A --> S
    S --> B
    B --> M
```

El bot no administra órdenes ni se conecta a PostgreSQL. Se conserva el nombre de la carpeta original para evitar romper rutas, pero el ejemplo usa MemoryDB para historial temporal de conversación. Ese historial se pierde al reiniciar y no sustituye auditoría, sesiones persistentes ni expedientes del negocio. La dependencia postgres de la plantilla se retiró porque el ejemplo usa MemoryDB.

## Plan por Sprint alineado al IEEE

| Sprint / fechas 2026 | Trabajo chatbot propuesto | Dependencia y evidencia |
| --- | --- | --- |
| 4: 5–11 octubre | Acordar contrato, formato de folio y verificación; preparar Meta, variables y flujo demo | Backend define orden/folio; diagrama y conversación demo |
| 5: 12–18 octubre | Preparar cliente HTTP, respuestas públicas y errores; acordar token del bot | Backend implementa estados y autorización por propietario; pruebas del contrato |
| 6: 19–25 octubre | Integrar RF-17/RF-19 con API real; validar contra consulta web | Endpoint real; evidencia de servicio propio, ajeno, inexistente y Backend caído |
| 7: 26–30 octubre | Integración y regresión; apoyar envío por evento Listo | Evento RF-18 desde Backend; demo beta para entrega del 30 octubre |
| 8: 2–8 noviembre | Robustez, límites de consulta, webhook, privacidad y fallos de canal | Pruebas negativas; no perder cambios de estado |
| 9: 9–15 noviembre | Usabilidad, configuración y FAQ si se prioriza HU-22 | Catálogo autorizado y manual; apoyo a respaldo/restauración |
| 10: 16–22 noviembre | Medir latencia, regresión y reunir evidencias de calidad | Mediciones RNF-01 y defectos corregidos |
| 11: 23–29 noviembre | Aceptación completa, instalación y demo final | Manual y evidencias para entrega del 30 noviembre |

Las tareas específicas del chatbot son una propuesta; las fechas y entregas proceden del IEEE. La entrega 1 del 1 octubre ya pasó en la fecha de esta revisión (4 octubre); no se da por completada sin evidencias.

## Decisiones pendientes antes de activar consultas reales

1. Backend y equipo deben definir cómo verificar al cliente. El ejemplo PROPONE comparar el remitente de WhatsApp con el teléfono verificado del propietario. No es una decisión resuelta por los PDFs; si no basta, agregar OTP u otro mecanismo acordado.
2. Acordar formato del folio de orden. El Backend actual genera `EQ-001` al registrar equipos; un folio de equipo no equivale automáticamente a una orden de servicio.
3. Implementar el endpoint descrito en CONTRATO_API.ipynb: actualmente no existe. No reutilizar el listado interno de equipos, que exige rol recepción/administrador y no está diseñado para consultas del cliente.
4. Definir credenciales del servicio bot con permisos de consulta y su rotación. El token propuesto no es el JWT del proveedor Meta ni equivale a verificar al cliente.
5. Confirmar campos públicos, catálogo FAQ y política de notificaciones. Antes de implementar avisos proactivos, verificar las reglas vigentes y plantillas en documentación oficial de Meta.

## Definición de terminado

Código comentado y guía actualizada; lint, compilación y pruebas del contrato aprobadas; conversación real por Meta registrada sin credenciales ni teléfonos personales en evidencias; consulta web y bot coincidentes; servicio ajeno/inexistente indistinguibles; fallos controlados. El ejemplo demo no satisface por sí solo RF-19: falta integración con el Backend real.

Fuentes técnicas consultadas: [Meta](https://www.builderbot.app/en/providers/meta), [funciones](https://www.builderbot.app/en/add-functions), [modularización](https://www.builderbot.app/en/showcases/modularize). El código se verifica contra los paquetes instalados de BuilderBot 1.4.2; no se actualizan dependencias en este trabajo.


# Contrato propuesto de consulta — pendiente de implementar

`POST {BACKEND_URL}/chatbot/servicios/consulta`, con `Authorization: Bearer <BACKEND_BOT_TOKEN>` y `Content-Type: application/json`.

Petición:

```json
{"folio":"FIX-001","telefono":"5213312345678"}
```

`telefono` proviene de `ctx.from` del proveedor Meta, no de un campo capturado. Backend valida el token del bot, normaliza el teléfono conforme al esquema acordado y verifica que pertenece al propietario del servicio. La comparación propuesta requiere un teléfono previamente verificado y no reemplaza un mecanismo adicional si el equipo lo exige. Antes de producción verificar autenticidad del webhook, límites de intentos y tratamiento de números reasignados.

Respuesta 200 (únicamente después de autorizar):

```json
{"folio":"FIX-001","estado":"En revisión","equipo":"Laptop Lenovo","actualizadoEn":"2026-10-04T12:00:00-06:00"}
```

| Código | Significado para el cliente HTTP |
| --- | --- |
| 200 | Validar DTO y mostrar sólo campos permitidos |
| 403 / 404 | Misma respuesta conversacional: no se pudo verificar el servicio |
| 401 | Credencial del bot inválida: consulta no disponible |
| 429 / 5xx | Consulta no disponible; sin reintentos automáticos |
| Timeout / red / JSON inválido | Consulta no disponible |

No retornar notas internas, domicilio, teléfonos ni datos de otros clientes. El bot rechaza estados desconocidos, folios distintos al solicitado y fechas inválidas. Backend es responsable de propiedad, límites de consulta y auditoría; nunca aceptar sólo el folio como autorización. Usar HTTPS fuera del entorno local.

No se ha implementado este endpoint ni modificado Backend en este ejemplo. Los mocks de pruebas sólo verifican el cliente HTTP, no demuestran autorización real.


# Guía del flujo modelo: consulta de reparación

Esta guía explica el código existente para que puedas construir los demás flujos. El plan está en `PLAN_CHATBOT.ipynb`; el contrato HTTP propuesto, en `CONTRATO_API.ipynb`. Las referencias a HU/RF y fechas de ese plan están pendientes de contrastar nuevamente: los tres PDF originales no están disponibles en la nueva ruta del proyecto.

## Recorrido de la conversación

```mermaid
flowchart TD
    A[Cliente escribe estado] --> B[Bot solicita folio]
    B --> C{Respuesta}
    C -->|cancelar| D[Termina sin consultar]
    C -->|Formato inválido| E[Explica formato y vuelve a capturar]
    E --> C
    C -->|Folio válido| F[Consulta servicio con folio y remitente Meta]
    F --> G{Resultado}
    G -->|encontrado| H[Muestra campos públicos y termina]
    G -->|no_autorizado| I[Respuesta genérica y termina]
    G -->|no_disponible| J[Informa indisponibilidad y termina]
```

## Cómo leer cada archivo

1. **`src/config.ts`**: `requerida` comprueba variables sin revelar valores. `cargarConfiguracion` valida puerto y modo, prepara opciones de Meta y exige credenciales de Backend sólo en modo API. No configura la aplicación de Meta: eso se hace en su panel.
2. **`src/app.ts`**: selecciona el servicio demo o HTTP, crea `MetaProvider`, registra el flujo con `createFlow` y arranca el servidor. Es el lugar donde conectarás un flujo nuevo. `MemoryDB` conserva historial temporal; la información real del servicio pertenece al Backend.
3. **`src/flows/consulta.flow.ts`**: `crearFlujoConsulta` recibe la función `consultar` y la bandera demo. `addKeyword` define entradas; `addAnswer` muestra la pregunta y `capture: true` espera la respuesta. El callback usa `ctx.body` para el folio y `ctx.from` para el remitente. Primero permite cancelar, después valida y finalmente consulta. `fallBack` vuelve a pedir el dato; `endFlow` envía la respuesta final. Las opciones están descritas en la [documentación de funciones de BuilderBot](https://www.builderbot.app/en/add-functions).
4. **`src/services/consultar-servicio.ts`**: define estados y datos públicos, normaliza el folio y valida respuestas externas. `crearConsultaApi` envía folio y teléfono con token del bot y timeout de cinco segundos. `leerServicio` selecciona los campos que se pueden mostrar. `crearConsultaDemo` devuelve un caso ficticio únicamente para el remitente configurado.
5. **`tests/consulta.test.ts`**: comprueba folios, selección de campos, restricción demo y manejo del transporte HTTP. Los mocks no verifican autorización real del Backend ni entrega por WhatsApp.

## Ejemplo para probar

Con `CHATBOT_MODE=demo`, variables de Meta completas y `DEMO_PHONE_NUMBER` configurado:

```text
Cliente: estado
Bot: Bienvenido a Fixify... Escribe tu folio o cancelar.
Cliente: DEMO-001
Bot: DATOS FICTICIOS — DEMOSTRACIÓN
     Folio: DEMO-001
     Equipo: Laptop de demostración
     Estado: En revisión
     Última actualización: 2026-10-04T12:00:00-06:00
```

Comprueba también `AB` (repite captura), `cancelar` (termina) y un folio diferente (respuesta genérica). En modo API comprueba Backend caído, respuesta inválida y servicio ajeno. El endpoint propuesto todavía requiere implementación e integración; no se considera finalizada la consulta real.

## Patrón para tu siguiente flujo

Antes de escribirlo, completa esta ficha:

| Parte | Qué documentar |
| --- | --- |
| Requisito | HU/RF y página del documento que lo sustenta |
| Entrada | Palabras que activan el flujo y datos solicitados |
| Validación | Formato, cancelación, reintentos y caducidad |
| Servicio | Contrato, identidad, permisos y respuesta pública |
| Salida | Mensajes de éxito, dato desconocido y fallo técnico |
| Evidencia | Conversación y pruebas; limitaciones pendientes |

Luego crea un servicio tipado y una fábrica de flujo en archivos separados. Inyecta el servicio desde `app.ts` y registra el flujo junto a `crearFlujoConsulta(...)` en `createFlow`. Usa palabras de entrada distintas. Si capturas varios datos, guarda sólo los necesarios en `state`, valida cada paso y elimina los datos al cancelar o concluir. Sigue el mismo orden: capturar → validar → consultar → responder → cerrar.

Documenta cada función con su propósito, entradas, resultado, errores y requisito asociado. En los comentarios interiores explica decisiones, por ejemplo por qué folio inexistente y servicio ajeno reciben el mismo mensaje. Evita duplicar en comentarios lo que ya expresa una línea de código.

## Comprobación del entorno

Ejecuta `pnpm test`, `pnpm lint` y `pnpm build` desde `chatbot/base-ts-meta-postgres`. Si los lanzadores buscan `node.exe` en Linux, las dependencias fueron copiadas desde otro sistema; reinstálalas en Linux con `pnpm install --frozen-lockfile`. No cambies el código del flujo para corregir ese problema de instalación.

La configuración del proveedor y webhook se describe en la [guía oficial de Meta en BuilderBot](https://www.builderbot.app/en/providers/meta). No uses sus ejemplos antiguos de versión o tarifas como configuración vigente de tu cuenta.


## Código comentado: src/config.ts

```typescript
/** Lee variables obligatorias sin incluir sus valores en mensajes de error. */
const requerida = (nombre: string): string => {
    const valor = process.env[nombre]?.trim()
    if (!valor) throw new Error(`Falta configurar ${nombre}`)
    return valor
}

/** Falla antes de iniciar si la configuración es incompleta; API es el modo predeterminado. */
export const cargarConfiguracion = () => {
    const modo = process.env.CHATBOT_MODE ?? 'api'
    if (!['api', 'demo'].includes(modo)) throw new Error('CHATBOT_MODE debe ser api o demo')
    const port = Number(process.env.PORT ?? 3008)
    if (!Number.isInteger(port) || port < 1 || port > 65535) throw new Error('PORT inválido')
    const meta = {
        jwtToken: requerida('META_ACCESS_TOKEN'), numberId: requerida('META_PHONE_NUMBER_ID'),
        verifyToken: requerida('META_VERIFY_TOKEN'), version: requerida('META_API_VERSION'),
    }
    const apiUrl = modo === 'api' ? requerida('BACKEND_URL') : ''
    if (modo === 'api' && !['http:', 'https:'].includes(new URL(apiUrl).protocol)) throw new Error('BACKEND_URL inválida')
    return { port, meta, demo: modo === 'demo', apiUrl,
        apiToken: modo === 'api' ? requerida('BACKEND_BOT_TOKEN') : '',
        telefonoDemo: modo === 'demo' ? requerida('DEMO_PHONE_NUMBER') : '',
    }
}

```


## Código comentado: src/app.ts

```typescript
import { createBot, createProvider, createFlow, MemoryDB } from '@builderbot/bot'
import { MetaProvider } from '@builderbot/provider-meta'
import { cargarConfiguracion } from './config'
import { crearFlujoConsulta } from './flows/consulta.flow'
import { crearConsultaApi, crearConsultaDemo } from './services/consultar-servicio'

/**
 * Punto de composición: configura Meta y conecta los flujos con sus servicios.
 * MemoryDB guarda el historial conversacional en memoria, no datos del negocio.
 * Las órdenes se consultan exclusivamente por Backend; no hay acceso a PostgreSQL.
 */
const main = async () => {
    const config = cargarConfiguracion()
    const consultar = config.demo
        ? crearConsultaDemo(config.telefonoDemo)
        : crearConsultaApi(config.apiUrl, config.apiToken)
    const provider = createProvider(MetaProvider, config.meta)
    const { httpServer } = await createBot({
        flow: createFlow([crearFlujoConsulta(consultar, config.demo)]),
        provider, database: new MemoryDB(),
    })
    // El proveedor registra el webhook de Meta. No se exponen las rutas administrativas de la plantilla.
    httpServer(config.port)
}

// Una configuración inválida termina el proceso para evitar un bot parcialmente iniciado.
main().catch((error: unknown) => {
    console.error(error instanceof Error ? error.message : 'No se pudo iniciar el chatbot')
    process.exitCode = 1
})

```


## Código comentado: src/flows/consulta.flow.ts

```typescript
import { addKeyword, MemoryDB } from '@builderbot/bot'
import { MetaProvider } from '@builderbot/provider-meta'
import { ConsultarServicio, validarFolio } from '../services/consultar-servicio'

/**
 * Flujo modelo: HU-21 / RF-19, apoyado por HU-16/17 y RF-17.
 * Entrada -> captura de folio -> validación -> Backend -> respuesta pública -> cierre.
 * La dependencia se inyecta para usar el mismo diálogo en demo y con API real.
 */
export const crearFlujoConsulta = (consultar: ConsultarServicio, demo: boolean) =>
    addKeyword<MetaProvider, MemoryDB>(['hola', 'menu', 'menú', 'estado', 'consulta'], { sensitive: true })
        .addAnswer([
            'Bienvenido a Fixify. Aquí puedes consultar el estado de tu reparación.',
            ...(demo ? ['MODO DEMOSTRACIÓN: sólo datos ficticios. Folio de ejemplo: DEMO-001.'] : []),
            'Escribe tu folio o *cancelar* para salir.',
        ], { capture: true }, async (ctx, { fallBack, endFlow }) => {
            // Cancelar no consulta la API ni conserva información adicional en estado.
            if (ctx.body.trim().toLowerCase() === 'cancelar') return endFlow('Consulta cancelada. Escribe *estado* para empezar de nuevo.')
            const folio = validarFolio(ctx.body)
            // fallBack mantiene la captura en este paso hasta recibir un folio válido.
            if (!folio) return fallBack('Usa un folio de 3 a 40 caracteres: letras, números o guiones. También puedes escribir *cancelar*.')
            const resultado = await consultar(folio, ctx.from)
            if (resultado.tipo === 'no_autorizado') {
                return endFlow('No pudimos verificar un servicio con esos datos. Revisa tu folio o contacta a recepción. Escribe *estado* para reintentar.')
            }
            if (resultado.tipo === 'no_disponible') {
                return endFlow('La consulta no está disponible en este momento. Intenta después o contacta a recepción.')
            }
            const s = resultado.servicio
            // Se muestran exclusivamente los campos del DTO; no se vuelca el JSON del Backend.
            return endFlow([
                ...(demo ? ['DATOS FICTICIOS — DEMOSTRACIÓN'] : []),
                `Folio: ${s.folio}`, `Equipo: ${s.equipo}`, `Estado: ${s.estado}`,
                `Última actualización: ${s.actualizadoEn}`,
                'Escribe *estado* para otra consulta.',
            ].join('\n'))
        })

```


## Código comentado: src/services/consultar-servicio.ts

```typescript
/** Estados oficiales: RN-03. El bot sólo presenta datos; no cambia órdenes. */
export const ESTADOS = ['Recibido', 'En revisión', 'Reparando', 'Listo', 'Entregado'] as const
export type EstadoServicio = typeof ESTADOS[number]

/** DTO público propuesto: excluye notas técnicas, costos internos y datos personales. */
export interface ServicioPublico {
    folio: string
    estado: EstadoServicio
    equipo: string
    actualizadoEn: string
}
export type ResultadoConsulta =
    | { tipo: 'encontrado'; servicio: ServicioPublico }
    | { tipo: 'no_autorizado' }
    | { tipo: 'no_disponible' }
export type ConsultarServicio = (folio: string, telefono: string) => Promise<ResultadoConsulta>

/** Formato provisional, pendiente de acordar con Backend; no es un requisito del PDF. */
export const validarFolio = (texto: string): string | null => {
    const folio = texto.trim().toUpperCase()
    return /^[A-Z0-9][A-Z0-9-]{2,39}$/.test(folio) ? folio : null
}

/** Valida la respuesta externa y selecciona únicamente los campos públicos permitidos. */
export const leerServicio = (dato: unknown, folio: string): ServicioPublico | null => {
    if (!dato || typeof dato !== 'object') return null
    const s = dato as Record<string, unknown>
    if (s.folio !== folio || !ESTADOS.includes(s.estado as EstadoServicio) ||
        typeof s.equipo !== 'string' || !s.equipo.trim() || s.equipo.length > 160 ||
        typeof s.actualizadoEn !== 'string' || !Number.isFinite(Date.parse(s.actualizadoEn))) return null
    return { folio, estado: s.estado as EstadoServicio, equipo: s.equipo, actualizadoEn: s.actualizadoEn }
}

/**
 * Adaptador HTTP del contrato PROPUESTO (el endpoint aún no existe en Backend).
 * El teléfono proviene de ctx.from, nunca de una respuesta escrita por el cliente.
 * Backend debe autenticar al bot Y comprobar la propiedad del servicio (RN-08/RN-11).
 * Fallos, respuestas inválidas y timeout no generan estados inventados.
 */
export const crearConsultaApi = (url: string, token: string): ConsultarServicio => async (folio, telefono) => {
    try {
        const response = await fetch(new URL('chatbot/servicios/consulta', url.endsWith('/') ? url : `${url}/`), {
            method: 'POST',
            headers: { 'Content-Type': 'application/json', Authorization: `Bearer ${token}` },
            body: JSON.stringify({ folio, telefono }),
            signal: AbortSignal.timeout(5000),
            redirect: 'error',
        })
        // No revelar si el folio existe cuando la propiedad no coincide.
        if ([403, 404].includes(response.status)) return { tipo: 'no_autorizado' }
        if (!response.ok) return { tipo: 'no_disponible' }
        const servicio = leerServicio(await response.json(), folio)
        return servicio ? { tipo: 'encontrado', servicio } : { tipo: 'no_disponible' }
    } catch {
        // No registrar teléfonos, credenciales ni el contenido de la respuesta externa.
        return { tipo: 'no_disponible' }
    }
}

/** Fixture educativo, separado de producción; sólo acepta el remitente demo configurado. */
export const crearConsultaDemo = (telefonoDemo: string): ConsultarServicio => async (folio, telefono) => {
    if (folio !== 'DEMO-001' || telefono !== telefonoDemo) return { tipo: 'no_autorizado' }
    return { tipo: 'encontrado', servicio: {
        folio: 'DEMO-001', estado: 'En revisión', equipo: 'Laptop de demostración',
        actualizadoEn: '2026-10-04T12:00:00-06:00',
    } }
}

```


## Código comentado: tests/consulta.test.ts

```typescript
import { test } from 'node:test'
import assert from 'node:assert/strict'
import { crearConsultaApi, crearConsultaDemo, leerServicio, validarFolio } from '../src/services/consultar-servicio'
const publico = { folio: 'FIX-001', estado: 'En revisión', equipo: 'Laptop', actualizadoEn: '2026-10-04T12:00:00-06:00' }

test('normaliza folios y rechaza entradas inválidas', () => {
    assert.equal(validarFolio(' fix-001 '), 'FIX-001')
    for (const f of ['', 'AB', '../orden', 'A'.repeat(41)]) assert.equal(validarFolio(f), null)
})
test('el DTO elimina información privada y rechaza inconsistencias', () => {
    assert.deepEqual(leerServicio({ ...publico, notaInterna: 'privado' }, 'FIX-001'), publico)
    for (const s of [null, { ...publico, estado: 'Inventado' }, { ...publico, actualizadoEn: 'ayer' }, { ...publico, folio: 'OTRO' }]) {
        assert.equal(leerServicio(s, 'FIX-001'), null)
    }
})
test('demo restringe folio y remitente', async () => {
    const consultar = crearConsultaDemo('5213300000000')
    assert.equal((await consultar('DEMO-001', '5213300000000')).tipo, 'encontrado')
    assert.equal((await consultar('DEMO-001', 'otro')).tipo, 'no_autorizado')
    assert.equal((await consultar('OTRO', '5213300000000')).tipo, 'no_autorizado')
})
// Mock del transporte: verifica el cliente, no la autorización del Backend real.
test('API envía identidad/token y controla respuestas inválidas y fallos', async (t) => {
    const original = globalThis.fetch
    t.after(() => { globalThis.fetch = original })
    const consultar = crearConsultaApi('http://localhost/api/v1', 'token-prueba')
    globalThis.fetch = async (url, options) => {
        assert.equal(String(url), 'http://localhost/api/v1/chatbot/servicios/consulta')
        assert.deepEqual(JSON.parse(options.body as string), { folio: 'FIX-001', telefono: 'remitente-meta' })
        assert.equal((options.headers as Record<string, string>).Authorization, 'Bearer token-prueba')
        assert.ok(options.signal)
        return new Response(JSON.stringify({ ...publico, notaInterna: 'privado' }))
    }
    assert.deepEqual(await consultar('FIX-001', 'remitente-meta'), { tipo: 'encontrado', servicio: publico })
    for (const status of [403, 404, 401, 429, 500]) {
        globalThis.fetch = async () => new Response('', { status })
        assert.equal((await consultar('FIX-001', 'remitente-meta')).tipo,
            [403, 404].includes(status) ? 'no_autorizado' : 'no_disponible')
    }
    for (const body of ['no es JSON', JSON.stringify({ ...publico, estado: 'Inventado' })]) {
        globalThis.fetch = async () => new Response(body)
        assert.equal((await consultar('FIX-001', 'remitente-meta')).tipo, 'no_disponible')
    }
    globalThis.fetch = async () => { throw new DOMException('Timeout', 'TimeoutError') }
    assert.equal((await consultar('FIX-001', 'remitente-meta')).tipo, 'no_disponible')
})

```


## Dependencias Python en Linux

El freeze original incluía `pywinpty`, específico de Windows, y estaba codificado en UTF-16. Se convirtió a UTF-8 y se añadió el marcador `sys_platform == "win32"`. Para Linux usa el archivo dedicado:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements-linux.txt
python -m jupyter notebook docs/Documentacion_chatbot.ipynb
```

El frontend requiere Tk instalado por el gestor de paquetes de tu distribución (`python3-tk` en Debian/Ubuntu, `python3-tkinter` en Fedora). pip no instala esa biblioteca del sistema. La resolución completa de las dependencias Python debe comprobarse con la versión de Python del equipo.


## Límites de la entrega

El endpoint de consulta del Backend sigue siendo una propuesta. No se verificó envío real por Meta sin configurar credenciales. Las pruebas de transporte no prueban autorización del Backend. Las HU/RF y fechas del plan requieren revisión con los PDF originales.


## Contenedor Linux
Desde `chatbot/`: `docker build -f base-ts-meta-postgres/Dockerfile -t fixify-chatbot .` y `docker run --env-file base-ts-meta-postgres/.env -p 3008:3008 fixify-chatbot`. La imagen usa Node 22 y pnpm; no incorpora secretos. La construcción Docker queda pendiente si Docker no está disponible.


## Verificación realizada
Instalación pnpm completada en Linux; cuatro pruebas aprobadas; ESLint, Rollup y `tsc --noEmit --incremental false` aprobados. Se sustituyó `rollup-plugin-typescript2` por `@rollup/plugin-typescript` y se permitieron scripts de instalación de esbuild, sharp y ffmpeg Linux. Sintaxis de requirements y estructura del notebook verificadas. No se instaló el entorno Python completo ni se construyó Docker; tampoco se hicieron envíos reales por Meta.
